<a name="top"></a>
<img src="../images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

##### Module 2.6：更多关于 ChiselTest 的内容
**上一篇：[综合实战：FIR 滤波器](2.5_exercise.ipynb)**<br>
**下一篇：[生成器：参数](3.1_parameters.ipynb)**

## 动机
Chisel 团队一直在改进测试框架。"ChiselTest" 提供了以下改进。

- 适用于单元测试和系统集成测试
- 设计为可组合的抽象和分层结构
- 高度易用，通过使编写单元测试尽可能简单、无痛（避免样板代码和其他琐事）和有用来鼓励编写单元测试

### 计划中的功能
- 能够针对多种后端和仿真器（如果测试向量不是静态的，可能需要链接到 Scala；或者在综合到 FPGA 时，使用有限的测试构造 API 子集）
- 将包含在基础 chisel3 中，以避免打包和依赖性问题

## 环境设置

In [ ]:
val path = System.getProperty("user.dir") + "/../source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.experimental._
import chisel3.experimental.BundleLiterals._
import chisel3.tester._
import chisel3.tester.RawTester.test

>本训练营在使用 chisel 导入方面与你可能在其他地方看到的略有不同。`import chisel3.tester.RawTester.test` 引入的是专门为训练营设计的 `test(...)` 版本。

---
# 基本的 Tester 实现

ChiselTest 从与 iotesters 相同的基本操作开始。以下是旧版 iotesters 和新版 ChiselTest 之间基本功能的简要对比。

|        | iotesters             | ChiselTest            |
| :----  | :---                  | :---                |
| poke   | poke(c.io.in1, 6)     | c.io.in1.poke(6.U)    |
| peek   | peek(c.io.out1)       | c.io.out1.peek()      |
| expect | expect(c.io.out1, 6)  | c.io.out1.expect(6.U) |
| step   | step(1)               | c.io.clock.step(1)  |
| initiate | Driver.execute(...) { c => | test(...) { c => |

让我们先看看 2.1 中的简单直通模块。

In [ ]:
// Chisel Code, but pass in a parameter to set widths of ports
class PassthroughGenerator(width: Int) extends Module { 
  val io = IO(new Bundle {
    val in = Input(UInt(width.W))
    val out = Output(UInt(width.W))
  })
  io.out := io.in
}

使用旧风格，一个简单的测试看起来像这样：

```scala
val testResult = Driver(() => new Passthrough()) {
  c => new PeekPokeTester(c) {
    poke(c.io.in, 0)     // 将输入设置为值 0
    expect(c.io.out, 0)  // 确认输出正确地为 0
    poke(c.io.in, 1)     // 将输入设置为值 1
    expect(c.io.out, 1)  // 确认输出正确地为 1
    poke(c.io.in, 2)     // 将输入设置为值 2
    expect(c.io.out, 2)  // 确认输出正确地为 2
  }
}
assert(testResult)   // Scala Code: if testResult == false, will throw an error
println("SUCCESS!!") // Scala Code: if we get here, our tests passed!
```



In [ ]:
test(new PassthroughGenerator(16)) { c =>
    c.io.in.poke(0.U)     // Set our input to value 0
    c.io.out.expect(0.U)  // Assert that the output correctly has 0
    c.io.in.poke(1.U)     // Set our input to value 1
    c.io.out.expect(1.U)  // Assert that the output correctly has 1
    c.io.in.poke(2.U)     // Set our input to value 2
    c.io.out.expect(2.U)  // Assert that the output correctly has 2
}

>为了说明 ChiselTest 推进时钟的方式，我们可以在之前的示例中添加一些 `step` 操作。

In [ ]:
test(new PassthroughGenerator(16)) { c =>
    c.io.in.poke(0.U)     // Set our input to value 0
    c.clock.step(1)    // advance the clock
    c.io.out.expect(0.U)  // Assert that the output correctly has 0
    c.io.in.poke(1.U)     // Set our input to value 1
    c.clock.step(1)    // advance the clock
    c.io.out.expect(1.U)  // Assert that the output correctly has 1
    c.io.in.poke(2.U)     // Set our input to value 2
    c.clock.step(1)    // advance the clock
    c.io.out.expect(2.U)  // Assert that the output correctly has 2
}

---
## 上述示例中需要注意的事项

ChiselTest 的 `test` 方法需要的样板代码更少。以前 `PeekPokeTester` 的功能现在已经内置到流程中。

`poke` 和 `expect` 方法现在是每个单独的 `io` 元素的一部分。
这为测试器提供了重要的提示，以便更好地检查类型。
`peek` 和 `step` 操作现在也是 `io` 元素的方法。

另一个区别是，被 poke 和 expect 的值是 Chisel 字面量。
虽然这里看起来很简单，但它为更高级和更有趣的示例提供了更强的类型检查。
随着即将到来的指定 `Bundle` 字面量能力的改进，这一功能将得到进一步增强。



# 具有 Decoupled 接口的模块
在本节中，我们将介绍 tester2 的一些用于处理 `Decoupled` 接口的工具。
`Decoupled` 接收一个 Chisel 数据类型，并为其提供 `ready` 和 `valid` 信号。
ChiselTest 提供了一些很好的工具，用于自动化并可靠地测试这些接口。

## 一个队列示例
`QueueModule` 传递数据，其数据类型由 `ioType` 决定。`QueueModule` 内部有 `entries` 个状态元素，意味着它可以容纳那么多元素，然后才会产生背压。

In [ ]:
class QueueModule[T <: Data](ioType: T, entries: Int) extends MultiIOModule {
  val in = IO(Flipped(Decoupled(ioType)))
  val out = IO(Decoupled(ioType))
  out <> Queue(in, entries)
}

## EnqueueNow 和 expectDequeueNow
*ChiselTest* 有一些内置方法用于处理 IO 中带有 Decoupled 接口的电路。在这个示例中，我们将看到如何向 `queue` 插入和提取值。

| 方法 | 描述 |
| :---   | :---        |
| enqueueNow | 向一个 `Decoupled` 输入接口添加（入队）一个元素 |
| expectDequeueNow | 从一个 `Decoupled` 输出接口移除（出队）一个元素 |
---


>注意：需要一些必需的样板代码 `initSource`、`setSourceClock` 等，以确保在测试开始时 `ready` 和 `valid` 字段都正确初始化。

In [ ]:
test(new QueueModule(UInt(9.W), entries = 200)) { c =>
    // Example testsequence showing the use and behavior of Queue
    c.in.initSource()
    c.in.setSourceClock(c.clock)
    c.out.initSink()
    c.out.setSinkClock(c.clock)
    
    val testVector = Seq.tabulate(200){ i => i.U }

    testVector.zip(testVector).foreach { case (in, out) =>
      c.in.enqueueNow(in)
      c.out.expectDequeueNow(out)
    }
}

## EnqueueSeq 和 DequeueSeq
现在我们介绍两个新方法，它们将入队和出队操作合并为单个操作。

| 方法 | 描述 |
| :---   | :---        |
| enqueueSeq | 持续向一个 `Decoupled` 输入接口添加（入队）`Seq` 中的元素，一次一个，直到序列耗尽 |
| expectDequeueSeq | 从一个 `Decoupled` 输出接口移除（出队）元素，一次一个，并将每个元素与 `Seq` 中的下一个元素进行比较 |
---
> 注意：下面的示例可以正常工作，但是按照编写的方式，`enqueueSeq` 必须在 `expectDequeueSeq` 开始之前完成。如果 `testVector` 的大小大于队列深度，此示例将失败，因为队列会填满而无法完成 `enqueueSeq`。你可以自己尝试一下，看看失败是什么样子的。在下一节中，我们将展示如何正确构建此类测试。

In [ ]:
test(new QueueModule(UInt(9.W), entries = 200)) { c =>
    // Example testsequence showing the use and behavior of Queue
    c.in.initSource()
    c.in.setSourceClock(c.clock)
    c.out.initSink()
    c.out.setSinkClock(c.clock)
    
    val testVector = Seq.tabulate(100){ i => i.U }

    c.in.enqueueSeq(testVector)
    c.out.expectDequeueSeq(testVector)
}

> 上一节中另一个重要的收获是，我们刚刚看到的函数 `enqueueNow`、`enqueueSeq`、`expectDequeueNow` 和 `expectDequeueSeq` 并不是 ChiselTest 中复杂特殊处理的逻辑。相反，它们是 ChiselTest 鼓励你从 ChiselTest 原语构建的那种测试框架构建的示例。要了解这些方法是如何实现的，请查看 [TestAdapters.scala](https://github.com/ucb-bar/chisel-testers2/blob/d199c5908828d0be5245f55fce8a872b2afb314e/src/main/scala/chisel3/tester/TestAdapters.scala)。

# ChiselTest 中的 Fork 和 Join

在本节中，我们将介绍如何在单元测试中并发运行部分代码。为此，我们将引入 tester2 的两个新特性。

| 方法 | 描述 |
| :---   | :---        |
| fork   | 启动一个并发代码块，可以通过在前一个 fork 的代码块末尾追加 `.fork` 来添加额外的并发 fork |
| join | 将多个相关的 fork 重新合并回调用线程 |
---

在下面的示例中，两个 `fork` 被链式连接，然后 `join` 在一起。在第一个 `fork` 块中，`enqueueSeq` 将持续添加元素直到耗尽。第二个 `fork` 块将在每个周期数据可用时执行 `expectDequeueSeq`。

>由 fork 创建的线程按确定性顺序运行，很大程度上按照它们在代码中指定的顺序，并且某些依赖于其他线程的容易出错的操作用运行时检查来禁止。

In [ ]:
test(new QueueModule(UInt(9.W), entries = 200)) { c =>
    // Example testsequence showing the use and behavior of Queue
    c.in.initSource()
    c.in.setSourceClock(c.clock)
    c.out.initSink()
    c.out.setSinkClock(c.clock)
    
    val testVector = Seq.tabulate(300){ i => i.U }

    fork {
        c.in.enqueueSeq(testVector)
    }.fork {
        c.out.expectDequeueSeq(testVector)
    }.join()
}

## 将 Fork 和 Join 用于 GCD
在本节中，我们将使用 fork join 方法来实现*最大公约数* **GCD** 的测试。
让我们从定义 IO Bundle 开始。我们将在此处添加一些样板代码，以便使用 `Bundle` *字面量*。正如注释中所说，希望很快就能支持字面量支持代码的自动生成。

In [ ]:
class GcdInputBundle(val w: Int) extends Bundle {
  val value1 = UInt(w.W)
  val value2 = UInt(w.W)
}

In [ ]:
class GcdOutputBundle(val w: Int) extends Bundle {
  val value1 = UInt(w.W)
  val value2 = UInt(w.W)
  val gcd    = UInt(w.W)
}

现在让我们来看一个*Decoupled* 版本的 **GCD**。我们在这里稍作修改，使用了 `Decoupled` 包装器，它为输入和输出 Bundle 添加了 `ready` 和 `valid` 信号。`Flipped` 包装器接收默认创建为输出的 `Decoupled` `GcdInputBundle`，并递归地将每个字段转换为相反的方向。`Decoupled` 的 Bundle 参数中的数据元素被放置在顶层字段 `bits` 中。

In [ ]:
/**
  * Compute GCD using subtraction method.
  * Subtracts the smaller of registers x and y from the larger until register y is zero.
  * value input register x is then the Gcd
  * returns a packet of information with the two input values and their GCD
  */
class DecoupledGcd(width: Int) extends MultiIOModule {

  val input = IO(Flipped(Decoupled(new GcdInputBundle(width))))
  val output = IO(Decoupled(new GcdOutputBundle(width)))

  val xInitial    = Reg(UInt())
  val yInitial    = Reg(UInt())
  val x           = Reg(UInt())
  val y           = Reg(UInt())
  val busy        = RegInit(false.B)
  val resultValid = RegInit(false.B)

  input.ready := ! busy
  output.valid := resultValid
  output.bits := DontCare

  when(busy)  {
    // during computation keep subtracting the smaller from the larger
    when(x > y) {
      x := x - y
    }.otherwise {
      y := y - x
    }
    when(y === 0.U) {
      // when y becomes zero computation is over,
      // signal valid data to output if the output is ready
      output.bits.value1 := xInitial
      output.bits.value2 := yInitial
      output.bits.gcd := x
      output.valid := true.B
      busy := ! output.ready
    }
  }.otherwise {
    when(input.valid) {
      // valid data available and no computation in progress, grab new values and start
      val bundle = input.deq()
      x := bundle.value1
      y := bundle.value2
      xInitial := bundle.value1
      yInitial := bundle.value2
      busy := true.B
      resultValid := false.B
    }
  }
}

我们的测试看起来与之前的队列测试几乎相同。
但这里涉及更多内容，因为计算需要多个周期，所以输入入队过程会在计算每个 GCD 时被阻塞。
好消息是，测试端对于不同的 Decoupled 电路来说既简单又一致。

这里还引入了新的 Chisel3 `Bundle` 字面量表示法。考虑以下代码行：
```scala
new GcdInputBundle(16).Lit(_.value1 -> x.U, _.value2 -> y.U)
```
上面定义的 `GcdInputBundle` 有两个字段 `value1` 和 `value2`。
我们通过先创建一个 Bundle，然后调用它的 `.Lit` 方法来创建一个 Bundle 字面量。
该方法接受一个可变参数列表的键/值对，其中键（例如 `_.value`）是字段名，值（例如 x.U）是 Chisel 硬件字面量，Scala `Int` x 被转换为 Chisel `UInt` 字面量。
字段名前的 `_.` 是必需的，用于将名称值绑定到 Bundle 内部。

>这可能不是完美的表示法，但在广泛的开发讨论中，它被视为在最小化样板代码和 Scala 中可用的表示法限制之间的最佳平衡。


In [ ]:
test(new DecoupledGcd(16)) { dut =>
  dut.input.initSource().setSourceClock(dut.clock)
  dut.output.initSink().setSinkClock(dut.clock)

  val testValues = for { x <- 1 to 10; y <- 1 to 10} yield (x, y)
  val inputSeq = testValues.map { case (x, y) =>
    (new GcdInputBundle(16)).Lit(_.value1 -> x.U, _.value2 -> y.U)
  }
  val resultSeq = testValues.map { case (x, y) =>
    new GcdOutputBundle(16).Lit(_.value1 -> x.U, _.value2 -> y.U, _.gcd -> BigInt(x).gcd(BigInt(y)).U)
  }

  fork {
    dut.input.enqueueSeq(inputSeq)
  }.fork {
    for (expected <- resultSeq) {
      dut.output.expectDequeue(expected)
      dut.clock.step(5) // wait some cycles before receiving the next output to create backpressure
    }
  }.join()
}


---
# 你已完成！

[返回顶部。](#top)